# 04b — DistilBERT Fine-Tune on M4 (Apple Silicon MPS)

## Goal

Supervised fine-tune `distilbert-base-uncased` (66M params) on stratified Yelp reviews, then evaluate on Yelp/Amazon/IMDB test sets. Compare against the best prompting result from notebook 04a.

## Why DistilBERT

- Encoder-only classifier for a 5-class problem — the textbook right tool.
- 66M params → trains on M4 MPS in ~18 min for 10k × 3 epochs.
- Shows classical ML fundamentals alongside the prompting work.

## Protocol

1. **Train** on 10 000 stratified Yelp reviews (2 000 per class), 3 epochs, `batch=16`, `lr=2e-5`.
2. **Evaluate** on the same 500/300/300 Yelp/Amazon/IMDB eval splits used in notebook 04a.
3. **Compare** to DeepSeek V3.2 few-shot.

Training is run as a subprocess so the notebook stays responsive. Already-trained weights live at `scripts/distilbert_yelp/` and are reused if present.

### To retrain from scratch:

```bash
python scripts/distilbert_train.py --n 10000 --epochs 3 --out scripts/distilbert_yelp
```

In [1]:
import sys; sys.path.insert(0, '..')
import json, subprocess, time
from pathlib import Path
import pandas as pd
import torch

print('python:', sys.version.split()[0])
print('torch :', torch.__version__)
print('MPS available:', torch.backends.mps.is_available())

MODEL_DIR = Path('../scripts/distilbert_yelp')
print('model dir:', MODEL_DIR, 'exists:', MODEL_DIR.exists())

python: 3.13.3
torch : 2.11.0
MPS available: True
model dir: ../scripts/distilbert_yelp exists: True


## Training summary (reads `final_metrics.json` written by train script)

In [2]:
metrics_path = MODEL_DIR / 'final_metrics.json'
if metrics_path.exists():
    m = json.loads(metrics_path.read_text())
    print(json.dumps(m, indent=2))
else:
    print('No trained model found. Run:')
    print('  python scripts/distilbert_train.py --n 10000 --epochs 3 --out scripts/distilbert_yelp')

{
  "eval_loss": 0.9480347633361816,
  "eval_acc": 0.59,
  "eval_macro_f1": 0.5893320773169244,
  "eval_mae": 0.482,
  "eval_runtime": 5.4203,
  "eval_samples_per_second": 92.245,
  "eval_steps_per_second": 2.952,
  "epoch": 3.0
}


## Cross-domain eval (reads cached results/04b_distilbert.json if present)

In [3]:
ft_path = Path('../results/04b_distilbert.json')

def run_eval():
    cmd = [
        sys.executable, 'scripts/distilbert_eval.py',
        '--model', 'scripts/distilbert_yelp',
        '--sets', 'data/yelp_eval.jsonl', 'data/amazon_eval.jsonl', 'data/imdb_eval.jsonl',
        '--out', 'results/04b_distilbert.json',
    ]
    print('$', ' '.join(cmd))
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd='..')
    print(f'exit={proc.returncode}  elapsed={time.time()-t0:.1f}s')
    if proc.returncode != 0:
        print('STDERR:', proc.stderr[-600:])

if not ft_path.exists() and MODEL_DIR.exists():
    run_eval()

if ft_path.exists():
    ft = json.loads(ft_path.read_text())
    for name, r in ft.items():
        print(f"\n{name}: acc={r['accuracy']*100:.1f}%  f1={r['macro_f1']:.3f}  mae={r['mae']:.3f}")
else:
    print('no eval results yet')


yelp_eval: acc=59.2%  f1=0.590  mae=0.460

amazon_eval: acc=51.7%  f1=0.494  mae=0.617

imdb_eval: acc=51.0%  f1=0.653  mae=0.913


## Head-to-head: DistilBERT (FT) vs DeepSeek V3.2 (few-shot)

In [4]:
prompt_path = Path('../results/04a_domain_shift.json')
if ft_path.exists() and prompt_path.exists():
    ft = json.loads(ft_path.read_text())
    prompt = json.loads(prompt_path.read_text())
    rows = []
    for d in ['yelp', 'amazon', 'imdb']:
        ft_key = f'{d}_eval'
        if ft_key in ft and d in prompt:
            rows.append({
                'domain': d,
                'prompt_acc':     round(prompt[d]['accuracy']*100, 1),
                'distilbert_acc': round(ft[ft_key]['accuracy']*100, 1),
                'prompt_f1':      round(prompt[d]['macro_f1'], 3),
                'distilbert_f1':  round(ft[ft_key]['macro_f1'], 3),
                'prompt_mae':     round(prompt[d]['mae'], 3),
                'distilbert_mae': round(ft[ft_key]['mae'], 3),
                'winner_acc':     'distilbert' if ft[ft_key]['accuracy'] > prompt[d]['accuracy'] else 'prompt',
            })
    pd.DataFrame(rows)
else:
    print('missing result files; run training + eval first')

## Findings

- **Yelp (in-domain)**: prompt-LLM beats DistilBERT by ~7 pp. Expected — 10k train is small; the SFT ceiling isn't reached. Published benchmarks put full-data DistilBERT at ~64% on Yelp-5.
- **Amazon (near-domain)**: prompt-LLM wins by ~12 pp. Encoder overfits Yelp-specific phrasing; LLM generalizes because it was never specialized.
- **IMDB (far + binary)**: DistilBERT wins by ~9 pp. Two likely causes:
  1. Encoder argmax concentrates mass at extremes — predicts 1★ or 5★ more often. That matches IMDB's binary truth.
  2. The LLM's ordinal prior hedges to the middle on ambiguous movie reviews; middle picks are penalized under binary labels.
- **MAE** tells the same story from a different angle: DistilBERT's IMDB MAE is worse (0.91) because when it's wrong it's *very* wrong, whereas the LLM stays closer.

## Trade-off takeaway

Classic ML lesson reproduces: encoder SFT wins in-domain, prompt-LLM wins out-of-domain — *but the crossover depends on training data size*. At 10k rows the LLM still beats in-domain; at 100k+, expect the flip. The far-domain IMDB surprise is a caveat to the 'LLM generalizes better' heuristic.

Next: `05_adversarial.ipynb`.